# Plain U-Net source: seed 42

Source implementation only: saved RGB Dice is 0.7459, which differs from the later author-confirmed final RGB result 0.7398. Final three-seed run code and exports still need to be attached. Do not label this historical output as the final manuscript run.

Source: `graindiscoloration (2).ipynb`, zero-based cell 76.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
# ============================================================
# PLAIN U-NET CONTROLLED MODALITY STUDY
# RGB-only vs MS-only vs RGB+MS Early Fusion
# One-cell Kaggle / Colab ready
# ============================================================

from __future__ import annotations

import csv
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Leave as None for automatic discovery.
# Kaggle example:
# BASE_DIR = "/kaggle/input/datasets/hemal7/gd-rgbms-db"
#
# Colab example:
# BASE_DIR = "/content/GD-RGBMS-DB"

BASE_DIR = None

WORK_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
)

OUT_DIR = WORK_DIR / "gd_plain_unet_modality_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# For quick conference experiment:
RUN_SEEDS = [42]

# If time permits later:
# RUN_SEEDS = [42, 123, 2026]

MODALITIES = ["RGB", "MS", "RGB_MS"]

TARGET_SIZE = (896, 672)   # PIL: width, height

BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
EARLY_STOPPING_PATIENCE = 12

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

# Plain U-Net feature widths
UNET_CHANNELS = [32, 64, 128, 256]

# Save probability maps and masks for later conference analyses.
SAVE_PROBABILITY_MAPS = True
SAVE_BINARY_MASKS = True

MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

# Existing train-split statistics from your previous experiment.
ALL_CHANNEL_NAMES = [
    "R", "G", "B",
    "Green", "Red-Edge", "NIR"
]

ALL_MEAN = np.array([
    0.41819795966148376,
    0.46573567390441895,
    0.2516770660877228,
    0.21822915971279144,
    0.16206900775432587,
    0.23585167527198792,
], dtype=np.float32)

ALL_STD = np.array([
    0.1540451943874359,
    0.1569177359342575,
    0.13375557959079742,
    0.07147330790758133,
    0.05124884843826294,
    0.04640943184494972,
], dtype=np.float32)

# Coarse thresholds during training.
TRAIN_THRESHOLDS = np.round(
    np.arange(0.20, 0.86, 0.05), 2
).tolist()

# Fine validation threshold after best checkpoint.
FINAL_THRESHOLDS = np.round(
    np.arange(0.20, 0.801, 0.01), 2
).tolist()

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


# ============================================================
# 2. DATASET DISCOVERY
# ============================================================

def read_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return [x.strip() for x in f if x.strip()]


def valid_dataset_root(path):
    path = Path(path)

    required = [
        path / "RGB",
        path / "MS",
        path / "Mark",
        path / "train_list.txt",
        path / "val_list.txt",
        path / "test_list.txt",
    ]

    if not all(p.exists() for p in required):
        return False

    try:
        return (
            len(read_list(path / "train_list.txt")) == 280
            and len(read_list(path / "val_list.txt")) == 60
            and len(read_list(path / "test_list.txt")) == 60
        )
    except Exception:
        return False


def find_dataset_root():

    search_roots = [
        Path("/kaggle/input"),
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]

    candidates = []

    for root in search_roots:

        if not root.exists():
            continue

        for train_file in root.rglob("train_list.txt"):

            candidate = train_file.parent

            if valid_dataset_root(candidate):
                candidates.append(candidate)

    if not candidates:

        raise FileNotFoundError(
            "\nDataset could not be found automatically.\n"
            "Set BASE_DIR manually near the top, for example:\n\n"
            'BASE_DIR = "/kaggle/input/datasets/hemal7/gd-rgbms-db"\n'
        )

    candidates.sort(
        key=lambda p: (
            "gd-rgbms" not in str(p).lower(),
            len(str(p))
        )
    )

    return candidates[0]


if BASE_DIR is None:
    BASE_DIR = find_dataset_root()

BASE_DIR = Path(BASE_DIR)

RGB_DIR = BASE_DIR / "RGB"
MS_DIR = BASE_DIR / "MS"
MARK_DIR = BASE_DIR / "Mark"

TRAIN_NAMES = read_list(BASE_DIR / "train_list.txt")
VAL_NAMES = read_list(BASE_DIR / "val_list.txt")
TEST_NAMES = read_list(BASE_DIR / "test_list.txt")

assert len(TRAIN_NAMES) == 280
assert len(VAL_NAMES) == 60
assert len(TEST_NAMES) == 60

assert not (set(TRAIN_NAMES) & set(VAL_NAMES))
assert not (set(TRAIN_NAMES) & set(TEST_NAMES))
assert not (set(VAL_NAMES) & set(TEST_NAMES))

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PIN_MEMORY = DEVICE.type == "cuda"

print("=" * 75)
print("PLAIN U-NET MODALITY COMPARISON")
print("=" * 75)
print("Dataset :", BASE_DIR)
print("Output  :", OUT_DIR)
print("Device  :", DEVICE)
print("Split   :", len(TRAIN_NAMES), len(VAL_NAMES), len(TEST_NAMES))
print("Modes   :", MODALITIES)
print("Seeds   :", RUN_SEEDS)
print("=" * 75)


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


# ============================================================
# 4. DATA LOADING
# ============================================================

def find_file(directory, stem, extensions):

    for ext in extensions:

        path = directory / f"{stem}{ext}"

        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find {stem} inside {directory}"
    )


def make_binary_mask(pil_image):

    arr = np.asarray(
        pil_image.convert("RGB"),
        dtype=np.int16
    )

    distance = np.abs(
        arr - MASK_POSITIVE_RGB
    ).sum(axis=2)

    return (distance < 80).astype(np.float32)


def read_full_sample(name):

    stem = Path(name).stem

    rgb_path = find_file(
        RGB_DIR,
        stem,
        [".JPG", ".jpg", ".JPEG", ".jpeg"]
    )

    ms_path = find_file(
        MS_DIR,
        stem,
        [".PNG", ".png"]
    )

    mark_path = find_file(
        MARK_DIR,
        stem,
        [".PNG", ".png"]
    )

    with Image.open(rgb_path) as img:
        rgb = img.convert("RGB").resize(
            TARGET_SIZE,
            BILINEAR
        )

    with Image.open(ms_path) as img:
        ms = img.convert("RGB").resize(
            TARGET_SIZE,
            BILINEAR
        )

    with Image.open(mark_path) as img:
        mark = img.convert("RGB").resize(
            TARGET_SIZE,
            NEAREST
        )

    rgb = (
        np.asarray(rgb, dtype=np.float32)
        / 255.0
    )

    ms = (
        np.asarray(ms, dtype=np.float32)
        / 255.0
    )

    mask = make_binary_mask(mark)

    return rgb, ms, mask, stem


def modality_information(mode):

    if mode == "RGB":

        return {
            "channels": ["R", "G", "B"],
            "mean": ALL_MEAN[:3],
            "std": ALL_STD[:3],
            "input_channels": 3,
        }

    elif mode == "MS":

        return {
            "channels": ["Green", "Red-Edge", "NIR"],
            "mean": ALL_MEAN[3:],
            "std": ALL_STD[3:],
            "input_channels": 3,
        }

    elif mode == "RGB_MS":

        return {
            "channels": ALL_CHANNEL_NAMES,
            "mean": ALL_MEAN,
            "std": ALL_STD,
            "input_channels": 6,
        }

    raise ValueError(mode)


class GDDataset(Dataset):

    def __init__(
        self,
        names,
        modality,
        augment=False,
    ):

        self.names = list(names)
        self.modality = modality
        self.augment = augment

        info = modality_information(modality)

        self.mean = info["mean"].reshape(
            1, 1, -1
        )

        self.std = info["std"].reshape(
            1, 1, -1
        )

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):

        name = self.names[index]

        rgb, ms, mask, stem = read_full_sample(name)

        if self.modality == "RGB":
            image = rgb

        elif self.modality == "MS":
            image = ms

        elif self.modality == "RGB_MS":
            image = np.concatenate(
                [rgb, ms],
                axis=2
            )

        else:
            raise ValueError(self.modality)

        # ----------------------------------------------------
        # IDENTICAL augmentation for every modality.
        # Only geometric augmentation is used so that
        # modality is the main experimental difference.
        # ----------------------------------------------------

        if self.augment:

            if random.random() < 0.5:

                image = np.flip(
                    image,
                    axis=1
                ).copy()

                mask = np.flip(
                    mask,
                    axis=1
                ).copy()

            if random.random() < 0.5:

                image = np.flip(
                    image,
                    axis=0
                ).copy()

                mask = np.flip(
                    mask,
                    axis=0
                ).copy()

        image = (
            (image - self.mean)
            / self.std
        ).astype(np.float32)

        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        mask = torch.from_numpy(
            mask[None]
        ).float()

        return image, mask, stem


def make_loaders(modality, seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        GDDataset(
            TRAIN_NAMES,
            modality,
            augment=True
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    val_loader = DataLoader(
        GDDataset(
            VAL_NAMES,
            modality,
            augment=False
        ),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    test_loader = DataLoader(
        GDDataset(
            TEST_NAMES,
            modality,
            augment=False
        ),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    return train_loader, val_loader, test_loader


# ============================================================
# 5. PLAIN U-NET
# No residual blocks
# No scSE
# No attention
# No modality-specific branches
# ============================================================

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class PlainUNet(nn.Module):

    def __init__(
        self,
        input_channels,
        channels=UNET_CHANNELS
    ):

        super().__init__()

        c1, c2, c3, c4 = channels

        self.pool = nn.MaxPool2d(2)

        self.enc1 = DoubleConv(
            input_channels,
            c1
        )

        self.enc2 = DoubleConv(
            c1,
            c2
        )

        self.enc3 = DoubleConv(
            c2,
            c3
        )

        self.enc4 = DoubleConv(
            c3,
            c4
        )

        self.bottleneck = DoubleConv(
            c4,
            c4 * 2
        )

        self.up4 = nn.ConvTranspose2d(
            c4 * 2,
            c4,
            kernel_size=2,
            stride=2
        )

        self.dec4 = DoubleConv(
            c4 * 2,
            c4
        )

        self.up3 = nn.ConvTranspose2d(
            c4,
            c3,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            c3 * 2,
            c3
        )

        self.up2 = nn.ConvTranspose2d(
            c3,
            c2,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            c2 * 2,
            c2
        )

        self.up1 = nn.ConvTranspose2d(
            c2,
            c1,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            c1 * 2,
            c1
        )

        self.out = nn.Conv2d(
            c1,
            1,
            kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        b = self.bottleneck(
            self.pool(e4)
        )

        d4 = self.up4(b)

        d4 = self.dec4(
            torch.cat([d4, e4], dim=1)
        )

        d3 = self.up3(d4)

        d3 = self.dec3(
            torch.cat([d3, e3], dim=1)
        )

        d2 = self.up2(d3)

        d2 = self.dec2(
            torch.cat([d2, e2], dim=1)
        )

        d1 = self.up1(d2)

        d1 = self.dec1(
            torch.cat([d1, e1], dim=1)
        )

        return self.out(d1)


# ============================================================
# 6. LOSS
# ============================================================

class ComboLoss(nn.Module):

    def forward(self, logits, targets):

        targets = targets.float()

        bce_map = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        bce = bce_map.mean()

        prob = torch.sigmoid(logits)

        pt = (
            prob * targets
            + (1.0 - prob) * (1.0 - targets)
        )

        focal = (
            bce_map
            * ((1.0 - pt) ** 2.0)
        ).mean()

        dims = tuple(
            range(1, logits.ndim)
        )

        intersection = (
            prob * targets
        ).sum(dims)

        denominator = (
            prob.sum(dims)
            + targets.sum(dims)
        )

        dice_loss = 1.0 - (
            (
                2.0 * intersection
                + 1e-6
            )
            /
            (
                denominator
                + 1e-6
            )
        ).mean()

        return (
            0.35 * bce
            + 0.25 * focal
            + 0.40 * dice_loss
        )


# ============================================================
# 7. METRICS
# ============================================================

def confusion_tensor(
    probabilities,
    masks,
    threshold
):

    pred = probabilities >= threshold
    target = masks >= 0.5

    tp = int((pred & target).sum())
    fp = int((pred & ~target).sum())
    fn = int((~pred & target).sum())
    tn = int((~pred & ~target).sum())

    return [tp, fp, fn, tn]


def confusion_numpy(
    probability,
    mask,
    threshold
):

    pred = probability >= threshold
    target = mask >= 0.5

    return [
        int(np.logical_and(pred, target).sum()),
        int(np.logical_and(pred, ~target).sum()),
        int(np.logical_and(~pred, target).sum()),
        int(np.logical_and(~pred, ~target).sum()),
    ]


def add_confusion(a, b):
    return [
        x + y
        for x, y in zip(a, b)
    ]


def calculate_metrics(values):

    tp, fp, fn, tn = values

    eps = 1e-7

    dice = (
        2 * tp + eps
    ) / (
        2 * tp + fp + fn + eps
    )

    iou = (
        tp + eps
    ) / (
        tp + fp + fn + eps
    )

    precision = (
        tp + eps
    ) / (
        tp + fp + eps
    )

    recall = (
        tp + eps
    ) / (
        tp + fn + eps
    )

    accuracy = (
        tp + tn + eps
    ) / (
        tp + fp + fn + tn + eps
    )

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
    }


def write_csv(path, rows):

    if len(rows) == 0:
        return

    with open(
        path,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=list(rows[0].keys())
        )

        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# 8. TRAIN ONE CONFIGURATION
# ============================================================

def train_one(modality, seed):

    print("\n\n")
    print("=" * 75)
    print(
        f"STARTING: {modality} | SEED {seed}"
    )
    print("=" * 75)

    set_seed(seed)

    info = modality_information(modality)

    train_loader, val_loader, test_loader = (
        make_loaders(modality, seed)
    )

    model = PlainUNet(
        input_channels=info["input_channels"]
    ).to(DEVICE)

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    print("Input channels :", info["channels"])
    print("Parameters     :", f"{parameter_count:,}")

    criterion = ComboLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    experiment_dir = (
        OUT_DIR
        / modality
        / f"seed_{seed}"
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    checkpoint_path = (
        experiment_dir
        / "best_model.pt"
    )

    history = []

    best_val_dice = -1.0
    best_epoch = 0

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        start_time = time.time()

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        train_loss = 0.0

        for step, (
            images,
            masks,
            _
        ) in enumerate(
            tqdm(
                train_loader,
                desc=(
                    f"{modality} "
                    f"seed {seed} "
                    f"epoch {epoch}"
                ),
                leave=False
            ),
            start=1
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            with torch.amp.autocast(
                "cuda",
                enabled=(
                    DEVICE.type == "cuda"
                )
            ):

                logits = model(images)

                loss = criterion(
                    logits,
                    masks
                )

                scaled_loss = (
                    loss
                    / ACCUMULATION_STEPS
                )

            scaler.scale(
                scaled_loss
            ).backward()

            train_loss += loss.item()

            if (
                step % ACCUMULATION_STEPS == 0
                or step == len(train_loader)
            ):

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

        train_loss /= len(train_loader)

        # ----------------------------------------------------
        # Validation for checkpoint selection
        # ----------------------------------------------------

        model.eval()

        threshold_totals = {
            t: [0, 0, 0, 0]
            for t in TRAIN_THRESHOLDS
        }

        val_loss = 0.0

        with torch.no_grad():

            for images, masks, _ in val_loader:

                images = images.to(
                    DEVICE,
                    non_blocking=True
                )

                masks = masks.to(
                    DEVICE,
                    non_blocking=True
                )

                with torch.amp.autocast(
                    "cuda",
                    enabled=(
                        DEVICE.type == "cuda"
                    )
                ):

                    logits = model(images)

                    loss = criterion(
                        logits,
                        masks
                    )

                val_loss += loss.item()

                probabilities = torch.sigmoid(
                    logits
                )

                for threshold in TRAIN_THRESHOLDS:

                    values = confusion_tensor(
                        probabilities,
                        masks,
                        threshold
                    )

                    threshold_totals[
                        threshold
                    ] = add_confusion(
                        threshold_totals[
                            threshold
                        ],
                        values
                    )

        validation_rows = []

        for threshold in TRAIN_THRESHOLDS:

            row = calculate_metrics(
                threshold_totals[
                    threshold
                ]
            )

            row["threshold"] = threshold

            validation_rows.append(row)

        best_validation = max(
            validation_rows,
            key=lambda x: x["dice"]
        )

        scheduler.step(
            best_validation["dice"]
        )

        minutes = (
            time.time()
            - start_time
        ) / 60.0

        history_row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": (
                val_loss
                / len(val_loader)
            ),
            "val_dice": best_validation[
                "dice"
            ],
            "val_iou": best_validation[
                "iou"
            ],
            "val_precision": best_validation[
                "precision"
            ],
            "val_recall": best_validation[
                "recall"
            ],
            "val_threshold": best_validation[
                "threshold"
            ],
            "learning_rate": (
                optimizer.param_groups[0][
                    "lr"
                ]
            ),
            "minutes": minutes,
        }

        history.append(history_row)

        print(
            f"{modality:6s} | "
            f"seed {seed} | "
            f"epoch {epoch:02d} | "
            f"train {train_loss:.4f} | "
            f"val Dice "
            f"{best_validation['dice']:.4f} | "
            f"IoU "
            f"{best_validation['iou']:.4f} | "
            f"th "
            f"{best_validation['threshold']:.2f}"
        )

        if (
            best_validation["dice"]
            > best_val_dice
        ):

            best_val_dice = (
                best_validation["dice"]
            )

            best_epoch = epoch

            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "seed": seed,
                    "modality": modality,
                    "channels": info["channels"],
                    "parameter_count": (
                        parameter_count
                    ),
                    "best_validation_dice": (
                        best_val_dice
                    ),
                },
                checkpoint_path
            )

        if (
            epoch - best_epoch
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                "Early stopping at "
                f"epoch {epoch}"
            )

            break

    write_csv(
        experiment_dir
        / "training_history.csv",
        history
    )

    # --------------------------------------------------------
    # Reload best checkpoint
    # --------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    model.eval()

    # --------------------------------------------------------
    # Collect probabilities
    # --------------------------------------------------------

    @torch.no_grad()
    def collect_predictions(loader):

        records = []

        for images, masks, stems in tqdm(
            loader,
            desc=f"Prediction {modality}"
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            with torch.amp.autocast(
                "cuda",
                enabled=(
                    DEVICE.type == "cuda"
                )
            ):

                logits = model(images)

                prob = torch.sigmoid(
                    logits
                )

            records.append(
                {
                    "stem": stems[0],
                    "mask": (
                        masks[0, 0]
                        .numpy()
                        .astype(np.float32)
                    ),
                    "probability": (
                        prob[0, 0]
                        .float()
                        .cpu()
                        .numpy()
                        .astype(np.float32)
                    ),
                }
            )

        return records

    val_records = collect_predictions(
        val_loader
    )

    test_records = collect_predictions(
        test_loader
    )

    # --------------------------------------------------------
    # Fine threshold selection from validation only
    # --------------------------------------------------------

    threshold_rows = []

    for threshold in FINAL_THRESHOLDS:

        total = [0, 0, 0, 0]

        for record in val_records:

            values = confusion_numpy(
                record["probability"],
                record["mask"],
                threshold
            )

            total = add_confusion(
                total,
                values
            )

        row = calculate_metrics(total)

        row["threshold"] = threshold

        threshold_rows.append(row)

    best_threshold_row = max(
        threshold_rows,
        key=lambda x: x["dice"]
    )

    selected_threshold = (
        best_threshold_row["threshold"]
    )

    write_csv(
        experiment_dir
        / "validation_threshold_search.csv",
        threshold_rows
    )

    # --------------------------------------------------------
    # Final TEST evaluation
    # --------------------------------------------------------

    global_total = [0, 0, 0, 0]

    per_image_rows = []

    for record in test_records:

        values = confusion_numpy(
            record["probability"],
            record["mask"],
            selected_threshold
        )

        global_total = add_confusion(
            global_total,
            values
        )

        row = calculate_metrics(values)

        gt_mask = (
            record["mask"] >= 0.5
        )

        prediction = (
            record["probability"]
            >= selected_threshold
        )

        row.update(
            {
                "stem": record["stem"],
                "gt_coverage_percent": (
                    float(
                        gt_mask.mean() * 100
                    )
                ),
                "predicted_coverage_percent": (
                    float(
                        prediction.mean() * 100
                    )
                ),
                "coverage_error_percent": (
                    float(
                        (
                            prediction.mean()
                            - gt_mask.mean()
                        )
                        * 100
                    )
                ),
                "absolute_coverage_error": (
                    float(
                        abs(
                            prediction.mean()
                            - gt_mask.mean()
                        )
                        * 100
                    )
                ),
            }
        )

        per_image_rows.append(row)

    global_metrics = calculate_metrics(
        global_total
    )

    global_metrics.update(
        {
            "modality": modality,
            "seed": seed,
            "threshold": selected_threshold,
            "parameter_count": (
                parameter_count
            ),
            "best_epoch": (
                checkpoint["epoch"]
            ),
        }
    )

    write_csv(
        experiment_dir
        / "test_metrics.csv",
        [global_metrics]
    )

    write_csv(
        experiment_dir
        / "per_image_test_metrics.csv",
        per_image_rows
    )

    # --------------------------------------------------------
    # Save prediction maps
    # --------------------------------------------------------

    if SAVE_PROBABILITY_MAPS:

        probability_dir = (
            experiment_dir
            / "probability_maps"
        )

        probability_dir.mkdir(
            exist_ok=True
        )

        for record in test_records:

            arr = np.round(
                np.clip(
                    record["probability"],
                    0,
                    1
                )
                * 65535
            ).astype(np.uint16)

            Image.fromarray(arr).save(
                probability_dir
                / f"{record['stem']}.png"
            )

    if SAVE_BINARY_MASKS:

        binary_dir = (
            experiment_dir
            / "binary_predictions"
        )

        binary_dir.mkdir(
            exist_ok=True
        )

        for record in test_records:

            arr = (
                record["probability"]
                >= selected_threshold
            ).astype(np.uint8) * 255

            Image.fromarray(arr).save(
                binary_dir
                / f"{record['stem']}.png"
            )

    # --------------------------------------------------------
    # Save config
    # --------------------------------------------------------

    configuration = {

        "architecture": "Plain U-Net",

        "residual_connections": False,
        "scse_attention": False,
        "attention": False,

        "modality": modality,

        "input_channels": info["channels"],

        "number_of_input_channels": (
            info["input_channels"]
        ),

        "fusion": (
            "Input-level concatenation"
            if modality == "RGB_MS"
            else "None"
        ),

        "dataset": str(BASE_DIR),

        "split": {
            "train": 280,
            "validation": 60,
            "test": 60,
        },

        "image_size": TARGET_SIZE,

        "seed": seed,

        "optimizer": "AdamW",

        "learning_rate": LEARNING_RATE,

        "weight_decay": WEIGHT_DECAY,

        "loss": (
            "0.35 BCE + "
            "0.25 focal + "
            "0.40 Dice"
        ),

        "augmentation": (
            "Random horizontal and "
            "vertical flips only"
        ),

        "threshold_selection": (
            "Validation set only"
        ),

        "test_time_augmentation": False,

        "parameter_count": (
            parameter_count
        ),
    }

    with open(
        experiment_dir / "config.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            configuration,
            f,
            indent=2
        )

    print("\nFINAL TEST RESULT")
    print("-" * 50)
    print("Mode      :", modality)
    print(
        "Threshold :",
        f"{selected_threshold:.2f}"
    )
    print(
        "Dice      :",
        f"{global_metrics['dice']:.4f}"
    )
    print(
        "IoU       :",
        f"{global_metrics['iou']:.4f}"
    )
    print(
        "Precision :",
        f"{global_metrics['precision']:.4f}"
    )
    print(
        "Recall    :",
        f"{global_metrics['recall']:.4f}"
    )

    del model
    del optimizer
    del scheduler
    del scaler

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return global_metrics


# ============================================================
# 9. RUN ALL THREE MODALITIES
# ============================================================

ALL_RESULTS = []

for seed in RUN_SEEDS:

    for modality in MODALITIES:

        result = train_one(
            modality,
            seed
        )

        ALL_RESULTS.append(result)


# ============================================================
# 10. SAVE COMPARISON TABLE
# ============================================================

write_csv(
    OUT_DIR / "all_results.csv",
    ALL_RESULTS
)

print("\n\n")
print("=" * 85)
print("FINAL PLAIN U-NET COMPARISON")
print("=" * 85)

print(
    f"{'Input':<10}"
    f"{'Seed':<8}"
    f"{'Dice':<12}"
    f"{'IoU':<12}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
    f"{'Threshold':<12}"
    f"{'Params'}"
)

print("-" * 85)

for row in ALL_RESULTS:

    print(
        f"{row['modality']:<10}"
        f"{row['seed']:<8}"
        f"{row['dice']:<12.4f}"
        f"{row['iou']:<12.4f}"
        f"{row['precision']:<12.4f}"
        f"{row['recall']:<12.4f}"
        f"{row['threshold']:<12.2f}"
        f"{row['parameter_count']:,}"
    )


# ============================================================
# 11. ZIP EVERYTHING
# ============================================================

zip_path = shutil.make_archive(
    str(
        WORK_DIR
        / "gd_plain_unet_modality_comparison"
    ),
    "zip",
    str(OUT_DIR)
)

print("\nAll experiments finished.")
print("Results folder:", OUT_DIR)
print("ZIP file      :", zip_path)

try:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
except Exception:
    pass